# 05 — Gqa / 05 Gqa

**Chapter 16 — File 3 of 5 / 第16章 — 第3个文件（共5个）**

---

## Summary / 总结

This script demonstrates **projection**.

本脚本演示 **projection**。

---
## Step 1 — Step 1

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GQA(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_kv_heads=None, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads or num_heads
        self.head_dim = hidden_dim // num_heads
        self.num_groups = num_heads // num_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, q, k, v, mask=None, rope=None):
        q_batch_size, q_seq_len, hidden_dim = q.shape
        k_batch_size, k_seq_len, hidden_dim = k.shape
        v_batch_size, v_seq_len, hidden_dim = v.shape

---
## Step 2 — projection

In [ ]:
q = self.q_proj(q) \
            .view(q_batch_size, q_seq_len, -1, self.head_dim) \
            .transpose(1, 2)
        k = self.k_proj(k) \
            .view(k_batch_size, k_seq_len, -1, self.head_dim) \
            .transpose(1, 2)
        v = self.v_proj(v) \
            .view(v_batch_size, v_seq_len, -1, self.head_dim) \
            .transpose(1, 2)

---
## Step 3 — apply rotary positional encoding

In [ ]:
if rope:
            q = rope(q)
            k = rope(k)

---
## Step 4 — compute grouped query attention

In [ ]:
q = q.contiguous()
        k = k.contiguous()
        v = v.contiguous()
        output = F.scaled_dot_product_attention(q, k, v,
                                                attn_mask=mask,
                                                dropout_p=self.dropout,
                                                enable_gqa=True)
        output = output.transpose(1, 2) \
                       .reshape(q_batch_size, q_seq_len, hidden_dim) \
                       .contiguous()
        output = self.out_proj(output)
        return output

---
## Learning Notes / 学习笔记

- **概念**: projection 是机器学习中的常用技术。  
  *projection is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Gqa / 05 Gqa
# Complete Code / 完整代码
# ===============================

import torch.nn as nn
import torch.nn.functional as F

class GQA(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_kv_heads=None, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads or num_heads
        self.head_dim = hidden_dim // num_heads
        self.num_groups = num_heads // num_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, q, k, v, mask=None, rope=None):
        q_batch_size, q_seq_len, hidden_dim = q.shape
        k_batch_size, k_seq_len, hidden_dim = k.shape
        v_batch_size, v_seq_len, hidden_dim = v.shape

        # projection
        q = self.q_proj(q) \
            .view(q_batch_size, q_seq_len, -1, self.head_dim) \
            .transpose(1, 2)
        k = self.k_proj(k) \
            .view(k_batch_size, k_seq_len, -1, self.head_dim) \
            .transpose(1, 2)
        v = self.v_proj(v) \
            .view(v_batch_size, v_seq_len, -1, self.head_dim) \
            .transpose(1, 2)

        # apply rotary positional encoding
        if rope:
            q = rope(q)
            k = rope(k)

        # compute grouped query attention
        q = q.contiguous()
        k = k.contiguous()
        v = v.contiguous()
        output = F.scaled_dot_product_attention(q, k, v,
                                                attn_mask=mask,
                                                dropout_p=self.dropout,
                                                enable_gqa=True)
        output = output.transpose(1, 2) \
                       .reshape(q_batch_size, q_seq_len, hidden_dim) \
                       .contiguous()
        output = self.out_proj(output)
        return output

---

➡️ **Next / 下一步**: File 4 of 5